# Lesson 16 - Deploying Scalable Agents with Microsoft Foundry

In this notebook you build a **production-ready customer support agent** for the fictional company **Contoso**. Unlike the earlier lessons, the point here is not the agent's reasoning loop — it is everything wrapped *around* it that makes an agent safe to run at scale:

1. **Tool calling** — order lookups and ticket creation.
2. **RAG** — policy answers from a knowledge base.
3. **Memory** — remembering the customer across turns.
4. **Model routing** — send simple requests to a small model, complex ones to a large model.
5. **Response caching** — serve repeated questions without a model call.
6. **Human approval** — refunds above a threshold pause for sign-off.
7. **Evaluation gate** — an offline test set that blocks a bad release.
8. **Observability** — OpenTelemetry tracing around every request.

Each section is self-contained and runnable. Read every line — the production primitives are kept deliberately small.


## Configurare

Înainte de a rula acest notebook, asigură-te că ai:

1. **Un proiect Microsoft Foundry** cu un model de chat implementat (de ex. `gpt-4.1-mini`).
2. **Conectat cu Azure CLI** — rulează `az login` în terminalul tău.
3. **Setat variabilele de mediu necesare:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint-ul proiectului tău Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — numele modelului tău implementat.

Secțiunea RAG folosește **Azure AI Search** atunci când `AZURE_SEARCH_SERVICE_ENDPOINT` și `AZURE_SEARCH_API_KEY` sunt setate, și trece la o căutare în memorie astfel încât notebook-ul să ruleze fără o resursă Search.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Unelte

Uneltele de producție execută muncă reală asupra sistemelor reale. Aici simulăm o bază de date de comenzi și un sistem de ticketing cu funcții simple Python. Decoratorul `@tool` le expune agentului.

Observați că `issue_refund` folosește `approval_mode="always_require"` pentru returnările de sume peste un prag — aceasta este primitiva „omul în buclă” pe care o implementăm mai târziu.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Baza de Cunoștințe a Politicii

Întrebările despre politică („care este perioada de returnare?”) ar trebui să fie răspunse dintr-o sursă autoritară, nu din memoria modelului. Împachetăm o mică bază de cunoștințe ca un instrument de căutare.

În producție, aceasta este **Azure AI Search**; aici oferim o căutare în memorie pe baza cuvintelor cheie astfel încât notebook-ul să ruleze oriunde și să comutăm automat la Azure AI Search atunci când variabilele de mediu sunt prezente.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Memorie

Un agent de suport care uită cu cine vorbește este un agent de suport prost. Păstrăm un magazin mic de profiluri per client și injectăm un rezumat scurt în instrucțiunile agentului. În producție, aceasta este un serviciu de memorie (vezi Lecția 13); aici un dicționar face modelul vizibil.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 și 5. Rutarea Modelului și Cache-ul Răspunsurilor

Două pârghii de cost legate într-un singur handler de cereri:

- **Rutarea**: un clasificator euristic ieftin decide dacă o cerere are nevoie de modelul mic sau de cel mare.
- **Cache-ul**: întrebările repetitive normalizate sunt servite direct din cache fără apel la model.

Clasificatorul de aici este intenționat simplu. În producție l-ați valida în raport cu traficul și ați putea înlocui cu Model Router-ul Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Agentul, Aprobarea Umană și Observabilitatea

Acum asamblăm agentul din instrumentele de mai sus și înfășurăm fiecare cerere într-un span OpenTelemetry. Funcția `handle_support_request` este gestionarul cererii în producție: cache → rutare → trasare → rulare → cache.

Aprobarea umană este gestionată de cadrul de lucru: deoarece `issue_refund` are `approval_mode="always_require"`, rularea se oprește și afișează o cerere de aprobare pe care o rezolvăm explicit.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Poarta de Evaluare

Aceasta este poarta de lansare din lecție: un set de teste offline evaluează agentul, iar implementarea continuă doar dacă rata de trecere depășește un prag. Evaluatorul aici este o verificare simplă a suprapunerii de cuvinte-cheie pentru a menține notebook-ul auto-suficient; în producție ați folosi un LLM ca judecător sau un evaluator de cadru (vezi Lecția 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Punând totul cap la cap: o lansare simulatã

Celula de mai jos arată întregul ciclu descris în lecție: rulează poarta de evaluare și "deploiează" doar dacă trece. Acesta este modelul pe care l-ai rula în CI înainte de a promova o versiune a agentului la Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Rezumat

Ai asamblat un agent de suport pentru clienți gata de producție cu toate preocupările operaționale integrate:

- **Unelte, RAG și memorie** oferă agentului capacitate și context.
- **Rutarea și caching-ul modelului** mențin latența și costurile sub control.
- **Aprobare umană** protejează acțiunile cu risc ridicat, cum ar fi rambursările mari.
- **Poarta de evaluare** blochează lansările proaste înainte de a fi implementate.
- **Urmărirea** face fiecare cerere observabilă.

### Provocare

Extinde acest agent pentru a:

1. **Suporta mai multe modele** — adaugă un al treilea nivel de „raționament” și direcționează escaladările/plângerile către acesta.
2. **Adaugă porți de evaluare** — extinde `TEST_CASES` pentru a include scenarii de aprobare a rambursării și confirmă că poarta prinde regresiile.
3. **Adaugă rutare conștientă de cost** — urmărește un cost estimat pe solicitare (mic vs mare vs cache) și tipărește un raport de cost după un lot de interogări mixte.

În următoarea lecție vei face drumul invers și vei rula un agent complet pe propriul tău calculator cu Microsoft Foundry Local și Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
